# Hands-On: Anomaly Detection on Wind-Turbine SCADA Data

A runnable companion to the *O&M in Wind Turbines with AI Methods* talk. The
**concepts** (ML, neural networks, autoencoders) are in the slides — here we
**build the detector**: an autoencoder that learns *normal* turbine behaviour
and flags deviations by their **reconstruction error**.

The pipeline in one line:

> **SCADA data → clean (normal only) → autoencoder → reconstruction loss → anomaly detection**

Each step has a short note explaining *what the cell does* and *what each helper
function does*, so you can follow along and tinker. Long/repetitive code lives in
the importable `tutorial_helpers` package, keeping the cells short and readable.

**Run it:** `Runtime → Run all`. The first cell pulls the code, data and a
**pre-trained model** from GitHub, so nothing here takes more than a few seconds.

### Setup
Clone the repository (helper code + dataset + pre-trained weights) and install
dependencies. After this, the working directory is the repo, so every path
below is relative to it.

In [ ]:
import os

# Idempotent: safe to re-run. Clones the repo for its code/data/weights,
# then cd's into it so every path below is relative to the repo root.
REPO = "wind-turbine-anomaly-tutorial"
if not os.path.exists(f"/content/{REPO}"):
    !git clone --depth 1 https://github.com/asmaletale/wind-turbine-anomaly-tutorial.git /content/{REPO}
%cd /content/{REPO}
!pip install -q -r requirements.txt

Import the scientific stack and **all tutorial helpers** at once
(`from tutorial_helpers import *`). The seeds make results reproducible.

In [3]:
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler

# Helpers: data loading/labelling/sequencing, plotting, and the model builder.
from tutorial_helpers import *

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("PyTorch", torch.__version__)

PyTorch 2.11.0+cpu


## 1 · Load the data

We read two tables straight from the cloned repo:

- **`df_full`** — SCADA sensor readings (temperatures, power, wind, …) sampled
  every 10 minutes, for several turbines. The `WTG` column says which turbine.
- **`df_log`** — the maintenance **fault log**: when something failed, on which
  turbine and component.

`pd.read_parquet` reads the compressed columnar files; `.head()` previews the
fault log.

In [4]:
df_full = pd.read_parquet("data/df_full_edp.prqt.gzip")
df_log  = pd.read_parquet("data/edp_log.prqt.gzip")

print(f"SCADA: {df_full.shape[0]:,} rows x {df_full.shape[1]} columns")
print(f"Fault log: {df_log.shape[0]} events")
df_log.head()

SCADA: 417,093 rows x 41 columns
Fault log: 28 events


,WTG,Component,Timestamp,Remarks
0,T01,GEARBOX,2016-07-18T02:10:00+00:00,Gearbox pump damaged
1,T06,GENERATOR,2016-07-11T19:48:00+00:00,Generator replaced
2,T06,GENERATOR,2016-07-24T17:01:00+00:00,Generator temperature sensor failure
3,T06,GENERATOR,2016-09-04T08:08:00+00:00,High temperature generator error
4,T06,GENERATOR,2016-10-27T16:26:00+00:00,Generator replaced


## 2 · Pick a test turbine and the features

We hold out **turbine T07** entirely — the model never sees it in training, so
evaluating on it mimics deployment on a brand-new turbine. To keep things
interpretable we model one component: the **HV transformer** (its 3 phase
temperatures).

**Helper — `get_feature_group(name)`**: looks up a named list of SCADA columns in
`feature_groups.json` (here `"TRANSFORMER"` → the 3 temperature columns), so we
don't hard-code column names.

In [ ]:
TEST_WT = "T07"
wtgs = df_full["WTG"].unique()
train_wtgs = wtgs[wtgs != TEST_WT]                 # everything except T07

group_features = get_feature_group("TRANSFORMER")  # 3 transformer temperatures
#group_features = get_feature_group("GENERATOR")

test_log = df_log[df_log["WTG"] == TEST_WT].copy()

print("Train turbines:", list(train_wtgs), "| Test turbine:", TEST_WT)
print("Features:", group_features)

Train turbines: ['T01', 'T06', 'T11'] | Test turbine: T07
Features: ['hv transformer phase 1 temperature - avg', 'hv transformer phase 2 temperature - avg', 'hv transformer phase 3 temperature - avg']


**Explore** the test turbine before modelling.

**Helper — `plot_scada_with_faults_only(scada_df, log_df)`**: draws an
interactive Plotly chart of the selected signals with each logged fault marked as
a red vertical line. Zoom in around a fault and look for the temperature behaving
unusually.

In [ ]:
fig = plot_scada_with_faults_only(
    df_full[df_full["WTG"] == TEST_WT][group_features], test_log)
fig.show()

## 3 · Label anomalies

The autoencoder must train on **normal data only** (otherwise it learns faults as
"normal" and later misses them). So we tag each timestamp normal (`1`) or
anomalous (`-1`), to *remove* anomalies from training and to *check* results
later.

**Helper — `add_anomaly_label_simple(df, log_df)`**: adds an `is_anomaly` column
using two rules — (1) a window around each logged fault (20 days before → 10
after), and (2) a physics check (little power while wind is strong). Returns a
copy of `df` with the new column. We apply it per turbine and concatenate.

In [ ]:
labelled = [add_anomaly_label_simple(df_full[df_full["WTG"] == w].copy(),
                                     df_log[df_log["WTG"] == w].copy())
            for w in wtgs]
df_labelled = pd.concat(labelled)

# A labelled copy of the test turbine, only for the plot below.
df_test_view = add_anomaly_label_simple(
    df_full[df_full["WTG"] == TEST_WT].copy(), test_log)

**Helper — `plot_anomaly_analysis(df, log_df, plot="scatter", x=, y=)`**:
draws the **power curve** (wind speed vs. active power) coloured by label.
Healthy points trace an S-curve; anomalies (red) fall off it — e.g. no power
despite strong wind.

In [ ]:
plot_anomaly_analysis(df_test_view, test_log, plot="scatter",
                      x="wind speed - avg", y="active power - avg")

## 4 · Preprocess and scale

Keep our feature columns (+ `WTG` and the label), fill the occasional gap with
the column mean, then **scale**. Neural nets train better when inputs share a
range.

We use **`RobustScaler`** (centre on the median, scale by the inter-quartile
range) because it resists the outliers we're hunting. Crucially we **fit on the
training turbines only**, then apply to the test turbine — no information leaks
from test to train.

In [ ]:
cols = group_features + ["WTG", "is_anomaly"]
df_clean = df_labelled[cols].dropna(axis=1, how="all").copy()
df_clean[group_features] = df_clean[group_features].fillna(df_clean[group_features].mean())

scaler = RobustScaler()
train_df = df_clean[df_clean["WTG"] != TEST_WT].copy()
test_df  = df_clean[df_clean["WTG"] == TEST_WT].copy()

train_df[group_features] = scaler.fit_transform(train_df[group_features])  # fit on train
test_df[group_features]  = scaler.transform(test_df[group_features])       # apply to test
print("train:", train_df.shape, "| test:", test_df.shape)

**Helper — `plot_scaling_effect(before_df, after_df, feature)`**: shows one
feature's distribution before vs. after scaling (histograms + box plot), to
confirm scaling worked without distorting the data.

In [ ]:
plot_scaling_effect(df_clean[df_clean["WTG"] != TEST_WT], train_df, group_features[0])

## 5 · Turn rows into sequences (sliding windows)

The autoencoder reads short **windows** of history, not single rows. We cut the
series into overlapping windows of **144 steps = 1 day** (10-min data), stepping
by a **stride** of 36 (~75% overlap → more training data). Then we drop any
window containing a labelled anomaly, so only *healthy* windows remain.

**Helpers:**
- `create_sequences(data, seq_length, stride)` → a 3-D array
  `(n_windows, seq_length, n_features)` of sliding windows.
- `anomaly_array_filtering(array)` → keeps only all-normal windows and strips
  the label column, returning a NumPy `float32` array.

In [ ]:
SEQUENCE_LENGTH = 144          # 1 day at 10-minute resolution
STRIDE = SEQUENCE_LENGTH // 4   # 36 -> ~75% overlap

seqs = []
for w in train_wtgs:
    t = train_df[train_df["WTG"] == w].drop(columns=["WTG"])
    arr = anomaly_array_filtering(create_sequences(t, SEQUENCE_LENGTH, STRIDE))
    if arr is not None:
        seqs.append(arr)

sequences = np.concatenate(seqs, axis=0)
print("Clean training sequences:", sequences.shape)   # (n_windows, 144, 3)

## 6 · Build the autoencoder and load the pre-trained weights

**Helper — `create_dense_autoencoder(input_dim, sequence_length, hidden_dim,
latent_dim)`**: builds a fully-connected PyTorch autoencoder (`DenseAutoencoder`,
a `nn.Module`) that flattens each window, compresses it through a
4-dimensional **bottleneck**, and reconstructs it.

**Helper — `plot_autoencoder_architecture(model)`**: visualises the layer
dimensions (the bottleneck) and parameter counts.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = create_dense_autoencoder(
    input_dim=len(group_features),
    sequence_length=SEQUENCE_LENGTH,
    hidden_dim=32,
    latent_dim=4,
).to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
plot_autoencoder_architecture(model)

Training for 100 epochs takes minutes, so we **load ready-made weights**
shipped in the repo (identical to live training: same data, same preprocessing).
`torch.load` + `load_state_dict` fills the model's parameters from the saved
`.pt` file.

In [ ]:
model.load_state_dict(
    torch.load("models/dense_ae_pretrained.pt",
               map_location=device, weights_only=True))
print("Pre-trained weights loaded ✓")

*Optional:* flip `DEMO_TRAIN` to `True` to fine-tune live for a few epochs and
watch the loss drop. No `compile` step needed — PyTorch uses a plain `Adam`
optimizer and `MSELoss` directly.

In [ ]:
DEMO_TRAIN = False
if DEMO_TRAIN:
    n_val = int(len(sequences) * 0.2)
    perm  = np.random.permutation(len(sequences))
    X_train = torch.tensor(sequences[perm[n_val:]], dtype=torch.float32)
    X_val   = torch.tensor(sequences[perm[:n_val]],  dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train, X_train),
                              batch_size=64, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val,   X_val), batch_size=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn   = nn.MSELoss()

    for epoch in range(1, 4):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss_fn(model(xb), yb).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            tr = np.mean([loss_fn(model(xb.to(device)), yb.to(device)).item()
                          for xb, yb in train_loader])
            vl = np.mean([loss_fn(model(xb.to(device)), yb.to(device)).item()
                          for xb, yb in val_loader])
        print(f"Epoch {epoch}/3  train={tr:.4f}  val={vl:.4f}")

## 7 · Detect anomalies

Run the held-out test turbine through the model and measure each window's
**reconstruction error** (mean squared error between input and output). High
error = the model couldn't reproduce it = likely anomaly.

`create_sequences(..., return_indexes=True)` also returns each window's end
timestamp, so we can index the errors by time. `model.predict` reconstructs the
windows; `model.encode` extracts the **latent codes** for further analysis. We
compute MSE per window and store it in `loss_df`.

In [ ]:
test_seq, idx = create_sequences(test_df[group_features], SEQUENCE_LENGTH,
                                 stride=SEQUENCE_LENGTH, return_indexes=True)
recon  = model.predict(test_seq)
latent = model.encode(test_seq)

loss = np.mean(np.square(test_seq - recon), axis=(1, 2))   # one error per window
loss_df = pd.DataFrame({"loss": loss}, index=idx)
print("Reconstruction errors:", loss_df.shape)
print(f"Loss — mean: {loss.mean():.4f}  max: {loss.max():.4f}")

**Helper — `plot_reconstruction_comparison(orig, recon, feature_indexes)`**:
overlays input (solid) vs. reconstruction (dashed) for a couple of windows. Tight
overlap = “looks normal to the model”.

In [ ]:
plot_reconstruction_comparison(
    test_seq, recon,
    show_features_index=list(range(len(group_features))),
    seq_timestamps=idx, log_df=test_log,
    feature_names=group_features)

**Helper — `plot_latent_space(latent, timestamps, log_df)`**: projects the
4-dimensional latent codes to 2D with PCA and colours each window by its
distance from the centroid. Windows that deviate in latent space often correspond
to fault periods.

In [ ]:
plot_latent_space(latent, idx, log_df=test_log)

**Helper — `plot_latent_space_projections(latent, timestamps, log_df)`**: shows
every pair of the 4 latent dimensions as 2-D scatter plots, coloured by time.
Useful for inspecting which bottleneck dimensions encode fault-related variation.

In [ ]:
plot_latent_space_projections(latent, idx, log_df=test_log)

**Helper — `plot_mahalanobis_distance(orig, recon, timestamps, log_df,
feature_names)`**: computes the Mahalanobis distance between each reconstructed
window and the training distribution — a **feature-aware** anomaly score that
accounts for correlations between the temperature channels.

In [ ]:
plot_mahalanobis_distance(test_seq, recon, idx, log_df=test_log,
                          feature_names=group_features)

**Helper — `plot_loss_distribution(df, features, loss_df, log_df,
percentile=)`**: the payoff plot — the error distribution, a threshold (96th
percentile), and the error over time with **real fault events** as red lines.
Where loss spikes line up with logged faults, the detector works.

In [ ]:
plot_loss_distribution(test_df, group_features, loss_df, test_log, percentile=0.96)

**Helper — `plot_df_with_log(loss_df, log_df, scada_df)`**: an interactive
two-panel Plotly figure — signals on top, reconstruction loss with fault
annotations below.

In [ ]:
fig = plot_df_with_log(loss_df, test_log, test_df[group_features],
                       output_name="ae_result")
fig.show()

## Wrap-up & next steps

You built an end-to-end anomaly detector in **PyTorch**: loaded SCADA + faults,
cleaned to *normal-only* data, windowed and scaled it, reconstructed with a
pre-trained autoencoder, and used the **reconstruction error**, **latent space**,
and **Mahalanobis distance** as complementary anomaly signals — all aligned with
real logged faults.

**Try next:** add more feature groups (`get_feature_group("GENERATOR")`, …), tune
the latent size or threshold, test other turbines, or fine-tune live
(`DEMO_TRAIN = True`). To retrain from scratch, see `scripts/train_and_save.py`.

*Thanks for coding along!* 🌬️⚡